# DCAT-AP-NL Requirement Analysis for Dataverse Metadata Exporter

* **[DCAT-AP-NL-DANS-reqs.md](DCAT-AP-NL-DANS-reqs.md)** - Technical Requirements analysis/scoping
 

## DCAT-AP & DCAT-AP-NL SHACL Shapes

*This notebook makes use of SPARQL queries agains the DCAT-AP & DCAT-AP-NL SHACL Shapes in order to understand the profiles (mandatory properties, cardinalaty, range of object properties, controlled-vocabulary used as values, etc.*


According to https://docs.geostandaarden.nl/dcat/dcat-ap-nl30/#17C1E0BE
> The SHACL rules of DCAT-AP-NL build on the SHACL rules from DCAT API-3.0. **All the rules from DCAT-AP-3.0 (see [DCAT AP-3.0 dcat-ap-SHACL.ttl](dcat-ap/releases/3.0.0/shacl/dcat-ap-SHACL.ttl)) are still applicable. DCAT-AP-NL only tightens some data rules.**

>To test whether a dataset description meets DCAT-AP-NL, it is also necessary to include both the DCAT-AP SHACL shapes and the DCAT-AP-NL SHACL shapes in the validation.

> To properly support the validation of the dataset descriptions, DCAT-AP breaks the SHACL shapes is split to support different validation scenarios and aspects. See the chapter **[Validation of DCAT-AP](https://semiceu.github.io/DCAT-AP/releases/3.0.0/#validation-of-dcat-ap)**.

**DCAT-AP-NL SHACL shapes** are divided into:

* [dcat-ap-nl/shapes/dcat-ap-nl-SHACL.ttl](dcat-ap-nl-SHACL.ttl): The SHACL shapes of DCAT-AP-NL, excluding the validation rules around the class range of properties.
* [dcat-ap-nl/shapes/dcat-ap-nl-SHACL-klassebereik.ttl](dcat-ap-nl-SHACL-klassebereik.ttl) The SHACL shapes of DCAT-AP-EN for validating the class range of properties, excluding the class range of properties with a value derived from a code list.
* [dcat-ap-nl/shapes/dcat-ap-nl-SHACL-klassebereik-codelijsten.ttl](dcat-ap-nl-SHACL-klassebereik-codelijsten.ttl:) dcat-ap-nl-SHACL class range-codelists.ttl : The SHACL shapes of DCAT-AP-EN for validating the class range of properties with a value from a code list.
* [dcat-ap-nl/shapes/dcat-ap-nl-SHACL-aanbevolen.ttl](dcat-ap-nl-SHACL-aanbevolen.ttl): The SHACL shapes of DCAT-AP-EN for validating recommended properties.

## Controlled Vocabularies Constraints

**[csvs/ap-nl-dataset-CVs.csv](csvs/ap-nl-dataset-CVs.csv)**  Overview of DCAT-AP Dataset properties with controlled vocs as range

[dcat-ap/releases/3.0.1/html/shacl/mdr-vocabularies.shape.ttl](dcat-ap/releases/3.0.1/html/shacl/mdr-vocabularies.shape.ttl) specifies the controlled vocabulary constraints on properties expressed by DCAT-AP in SHACL.

More info in [DCAT-AP Documentation on CVs](https://semiceu.github.io/DCAT-AP/releases/3.0.1/#controlled-vocabularies-to-be-used)

In [ ]:
# boiler plate functions
# imports SPARQL prefixes and functions defs
import csv
from pprint import pprint
# from SPARQLWrapper import SPARQLWrapper, JSON, TURTLE, CSV 
from rdflib import Graph

prefixes = '''    
PREFIX adms: <http://www.w3.org/ns/adms#>
PREFIX dct: <http://purl.org/dc/terms/>
PREFIX dcat: <http://www.w3.org/ns/dcat#>
PREFIX dcatap: <http://data.europa.eu/r5r/>
PREFIX eli: <http://data.europa.eu/eli/ontology#>
PREFIX eush: <https://purl.eu/ns/shacl#>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX prov: <http://www.w3.org/ns/prov#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX sh: <http://www.w3.org/ns/shacl#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX vcard: <http://www.w3.org/2006/vcard/ns#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

PREFIX dcatapnl-sh: <http://modellen.geostandaarden.nl/dcat-ap-nl/id/shape/>
'''    

def sparql_files_query(query, format, filespath):
    g = Graph()
    for filepath in filespath:
        g.parse(filepath, format=format)  # can also use "ttl" for Turtle
    query = prefixes + query
    results = g.query(query)
    return results

def create_csv(filepath, headers, data_dict):

    with open(filepath, 'w', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=headers)
        writer.writeheader()
        writer.writerows(data_dict)



In [ ]:
# DCAT-AP SHACL shapes targeting dcat:Dataset 
sparql_dataset_shapes = ''' 
DESCRIBE ?shape ?shape_prop
WHERE{
    BIND( dcat:Dataset AS ?targetClass)
    ?shape sh:targetClass ?targetClass ;
            sh:property ?shape_prop .     
}
'''
result_distrib_shapes = sparql_files_query(query=sparql_dataset_shapes,
                                           format='ttl', 
                                           filespath=['dcat-ap/releases/3.0.1/shacl/dcat-ap-SHACL.ttl'])
print(result_distrib_shapes.serialize(format='ttl').decode('utf-8'))

In [ ]:

# # DCAT-AP-NL SHACL shapes targeting dcat:Dataset 

result_distrib_shapes = sparql_files_query(query=sparql_dataset_shapes,
                                           format='ttl', 
                                           filespath=['dcat-ap-nl/shapes/dcat-ap-nl-SHACL.ttl'])
print(result_distrib_shapes.serialize(format='ttl').decode('utf-8'))

In [ ]:
# INVESTIGATION
# Goal: understand how the cardinality 1..(mandatory) is expressed in DCAT-AP shacl
# By: SPARQL DESCRIBE of dcat-ap-SHACL.ttl dcat:Dataset: dct:description  shape in DCAT-AP SHACL
# Answer: via property:value  shacl:minCount 1 ;

sparql_dcatap_dataset_1mandatory_props = ''' 
DESCRIBE ?prop_shape
WHERE {
    BIND(dct:description AS ?prop_path) .
    <https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DatasetShape> sh:property ?prop_shape .
    ?prop_shape sh:path ?prop_path .
}
'''
results = sparql_files_query(query=sparql_dcatap_dataset_1mandatory_props, 
                            format='ttl', 
                            filespath=['dcat-ap/releases/3.0.1/shacl/dcat-ap-SHACL.ttl'])
print(results.serialize(format='ttl').decode('utf-8'))


The pattern I see in the cell above (`dcat:Dataset: dct:description`) property shapes, makes me conclude that 
* in **DCAT-AP SHACL the required properties have `shacl:minCount 1`**, which makes sense

Follow-up questions/queries?

* which other Dataset properties have `shacl:minCount 1` AKA are mandatory? 
* is the same pattern present in DCAT-AP-NL shacl?

In [ ]:
# OUTPUT
# Goal: list of all dcat:Dataset mandatory properties in DCAT-AP & DCAT-APN-NL
# shacl:minCount 1

sparql_apnl_dataset_mandatory_props = ''' 
SELECT ?prop_path  ?prop_shape ?maxCount
WHERE {
    {   # DCAT-AP query
        <https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DatasetShape> sh:property ?prop_shape .
        ?prop_shape sh:minCount 1 ;
            sh:path ?prop_path .
    }
    UNION
    {   # DCAT-AP-NL query
        dcatapnl-sh:DatasetShape sh:property ?prop_shape .
        ?prop_shape sh:minCount 1 ;
            sh:path ?prop_path . 
    }
}
ORDER BY ?prop_path
'''


results_ap_nl = sparql_files_query(query=sparql_apnl_dataset_mandatory_props, 
                            format='ttl', 
                            filespath=[ 
                                'dcat-ap-nl/shapes/dcat-ap-nl-SHACL.ttl',
                                'dcat-ap/releases/3.0.1/shacl/dcat-ap-SHACL.ttl'])

print(f"{'*'*10} Dataset Mandatory Properties in DCAT-AP & DCAT-AP-NL  {'*'*10}")
for row in results_ap_nl:
    pprint(row.asdict())

ap_NL_mandatory_dataset_props_list = [row.asdict() for row in results_ap_nl]

create_csv(filepath='dcat-ap-nl_mand_props.csv',
           headers=ap_NL_mandatory_dataset_props_list[0].keys(),
           data_dict=ap_NL_mandatory_dataset_props_list)


# Requirement: Mandatory Dataset properties

The response to the query above, tells us that the mandatory Dataset properties are:

See [csvs/ap-nl-dataset-mand-props.csv](csvs/ap-nl-dataset-mand-props.csv) where this info is compiled

**DCAT-AP mandatory properties of dcat:Dataset:**

*  http://purl.org/dc/terms/description
*  http://purl.org/dc/terms/title 

**DCAT-AP-NL mandatory properties of dcat:Dataset:**

* http://purl.org/dc/terms/accessRights 
* http://www.w3.org/ns/dcat#contactPoint 
* http://purl.org/dc/terms/creator 
* http://purl.org/dc/terms/identifier 
* http://purl.org/dc/terms/publisher 
* http://www.w3.org/ns/dcat#theme


## Range of DCAT-AP + DCAT-AP-NL Mandatory Properties

The range is the type of values a property can have.

The focus here is to **find the ranges of *object properties* (that have other RDF nodes as their value)** in n [dcat-ap/releases/3.0.1/shacl/ranges.ttl](dcat-ap/releases/3.0.1/shacl/ranges.ttl). *Data properties*, that have strings or numbers as values, are not being address by ranges.ttl.


Example of Dataset dcat:creator property and its range description defined by `shacl:class foaf:Agent`

```
<https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DatasetShape/c1d40f7102c8201949576e76be48b991b47958d9> rdfs:seeAlso "https://semiceu.github.io/DCAT-AP/releases/3.0.1#Dataset.creator";
  shacl:class foaf:Agent;
  shacl:description "An entity responsible for producing the dataset."@en;
  shacl:name "creator"@en;
  shacl:path dc:creator .
```




In [ ]:
# Goal: info **Ranges** of Dataset mandatory object properties
# note: object properties have as values RDF nodes (class instances), in contrast with data properties which have literals as values
# Output: Dataset mandatory object properties shapes - 
# property in schal:path; range in shacl:class
#  

prop_path_for_sparql = [item['prop_path'] for item in ap_NL_mandatory_dataset_props_list]
prop_path_for_sparql_str4query = (' '.join([f'<{str(uri)}>' for uri in prop_path_for_sparql]))
# use prop_path_for_sparql_str4query as VALUES in following range query
sparql_dcat_mandatory_dataset_props_ranges = '''
DESCRIBE  ?prop_shape_uri
WHERE {
    BIND( dcat:Dataset AS ?targetClass )
    VALUES ?prop_path { %s } 
    ?shape a sh:NodeShape ;
           sh:targetClass ?targetClass ;
           sh:property ?prop_shape_uri .
    ?prop_shape_uri sh:path ?prop_path  .
    
}''' % prop_path_for_sparql_str4query
print(f"{'*'*3} Querying shapes for Dataset mandatory propreties: {prop_path_for_sparql_str4query} {'*'*3}\n")
describe_dcatap_NL_mandatory_dataset_props = sparql_files_query(query=sparql_dcat_mandatory_dataset_props_ranges, 
                                                          format='ttl', 
                                                          filespath=['dcat-ap/releases/3.0.1/shacl/ranges.ttl'])
print(describe_dcatap_NL_mandatory_dataset_props.serialize(format='ttl').decode('utf-8'))

# TODO: include this info in the CSV dcat-ap-nl_mand_props.csv

In [ ]:
# continuation of previous cell 
# Output:  for DCAT-AP-NL Dataset mandatory properties and their range (sh:class ?prop_range) 
# Output: ap-nl-dataset-mand-props.csv

print(f'{"-"*40}\nQuerying in dcat-ap/releases/3.0.1/shacl/ranges.ttl Dataset property shapes:\n{prop_path_for_sparql_str4query}\n{"-"*40}')

sparql_dcat_mandatory_distribution_props_ranges_vars = '''
SELECT  ?prop_path  ?prop_range
WHERE {
    BIND( dcat:Dataset AS ?targetClass )
    VALUES ?prop_path { %s } 
    ?shape a sh:NodeShape ;
           sh:targetClass ?targetClass ;
           sh:property ?prop_shape_uri .
    ?prop_shape_uri sh:path ?prop_path ;
                    sh:class ?prop_range .
    
}''' % prop_path_for_sparql_str4query


ap_NL_mandatory_dataset_props_range = sparql_files_query(query=sparql_dcat_mandatory_distribution_props_ranges_vars, 
                                                          format='ttl', 
                                                          filespath=['dcat-ap/releases/3.0.1/shacl/ranges.ttl'])
ap_NL_mandatory_dataset_props_range_list = [row.asdict() for row in ap_NL_mandatory_dataset_props_range]

# Join  ap_NL_mandatory_dataset_props_range_list & ap_NL_mandatory_dataset_props_list
# Step 1: Build a mapping from prop_path to prop_range
prop_range_map = {d['prop_path']: d['prop_range'] for d in ap_NL_mandatory_dataset_props_range_list}
# Step 2: Merge the lists
ap_NL_mandatory_dataset_props_merged = []
for d in ap_NL_mandatory_dataset_props_list:
    # Copy to avoid mutating the original
    merged_dict = d.copy()
    prop_path = d['prop_path']
    if prop_path in prop_range_map:
        merged_dict['prop_range'] = prop_range_map[prop_path]
    ap_NL_mandatory_dataset_props_merged.append(merged_dict)
print(f"{'*'*3} Output of ap_NL_mandatory_dataset_props_merged in ap-nl-dataset-mand-props.csv {'*'*3}\n")
# print(ap_NL_mandatory_dataset_props_merged)
create_csv(filepath='csvs/ap-nl-dataset-mand-props.csv',
           headers=['prop_path', 'prop_range', 'prop_shape'],
           data_dict=ap_NL_mandatory_dataset_props_merged)


In [ ]:
%%!
echo "---- DCAT-AP-NL - mandatory props range ------"
csvtk pretty csvs/ap-nl-dataset-mand-props.csv


# Recommended properties of Dataset

in [csvs/ap-nl-dataset-recommended-props.csv](csvs/ap-nl-dataset-recommended-props.csv)


In [ ]:
# goal: list of recommended properties
# output: csvs/ap-nl-dataset-recommended-props.csv
sparql_dataset_props_cvs = '''
SELECT ?dataset_prop ?voc_desc
WHERE {
       <http://data.europa.eu/r5r#Dataset_ShapeCV> sh:property ?shape_prop .
       ?shape_prop sh:path ?dataset_prop ;
                   sh:description ?voc_desc  
          
}''' 


ap_dataset_props_CVs = sparql_files_query(query=sparql_dataset_props_cvs, 
                                                          format='ttl', 
                                                          filespath=['dcat-ap/releases/3.0.1/html/shacl/mdr-vocabularies.shape.ttl'])
for result in ap_dataset_props_CVs:
    print(result)

ap_dataset_props_CVs_list = [item.asdict() for item in ap_dataset_props_CVs]
print(ap_dataset_props_CVs_list)
create_csv(filepath='csvs/ap-nl-dataset-CVs.csv',
           headers=['dataset_prop', 'voc_desc'],
           data_dict=ap_dataset_props_CVs_list
           )    

In [ ]:
# Goal: recommended Dataset properties in DCAT-AP & DCAT-AP-NL - SHACLS

describe_dataset_rec_props = '''
DESCRIBE ?prop_shape
WHERE {
   BIND( dcat:Dataset AS ?targetClass)
  ?shape sh:targetClass ?targetClass ;
         sh:property ?prop_shape  . 
}
'''

results_dataset_rec_props = sparql_files_query(query=describe_dataset_rec_props,
                                format='ttl', 
                                filespath=['dcat-ap/releases/3.0.1/html/shacl/shapes_recommended.ttl',
                                           'dcat-ap-nl/shapes/dcat-ap-nl-SHACL-aanbevolen.ttl'])
print(results_dataset_rec_props.serialize(format='ttl').decode('utf-8'))



In [ ]:
# Goal: recommended Dataset properties in DCAT-AP & DCAT-AP-NL - list
# output csvs/ap-nl-dataset-recommended-props.csv


'''
SELECT ?dataset_prop ?profile
WHERE {
        { 
            BIND('ap' AS ?profile)
            <http://data.europa.eu/r5r#Dataset_Shape> sh:property ?shape_prop .
            ?shape_prop sh:path ?dataset_prop .
        }
        UNION
        { 
            BIND('ap-NL' AS ?profile)
            <http://modellen.geostandaarden.nl/dcat-ap-nl/id/shape/DatasetShape_aanbevolen> sh:property ?shape_prop .
            ?shape_prop sh:path ?dataset_prop .
        }        
}''' 


ap_dataset_recommended_props = sparql_files_query(query=sparql_dataset_rec_props, 
                                                          format='ttl', 
                                                          filespath=['dcat-ap/releases/3.0.1/html/shacl/shapes_recommended.ttl',
                                                                     'dcat-ap-nl/shapes/dcat-ap-nl-SHACL-aanbevolen.ttl'
                                                                     ])
for result in ap_dataset_recommended_props:
    print(result)

ap_dataset_recommended_props_list = [item.asdict() for item in ap_dataset_recommended_props]
create_csv(filepath='csvs/ap-nl-dataset-recommended-props.csv',
           headers=['dataset_prop', 'profile'],
           data_dict=ap_dataset_recommended_props_list
           )   

In [ ]:
# Goal: recommended Dataset properties in DCAT-AP & DCAT-AP-NL 


sparql_dataset_rec_props = '''
SELECT ?dataset_prop ?profile
WHERE {
        { 
            BIND('ap' AS ?profile)
            <http://data.europa.eu/r5r#Dataset_Shape> sh:property ?shape_prop .
            ?shape_prop sh:path ?dataset_prop .
        }
        UNION
        { 
            BIND('ap-NL' AS ?profile)
            <http://modellen.geostandaarden.nl/dcat-ap-nl/id/shape/DatasetShape_aanbevolen> sh:property ?shape_prop .
            ?shape_prop sh:path ?dataset_prop .
        }        
}''' 


ap_dataset_recommended_props = sparql_files_query(query=sparql_dataset_rec_props, 
                                                          format='ttl', 
                                                          filespath=['dcat-ap/releases/3.0.1/html/shacl/shapes_recommended.ttl',
                                                                     'dcat-ap-nl/shapes/dcat-ap-nl-SHACL-aanbevolen.ttl'
                                                                     ])
for result in ap_dataset_recommended_props:
    print(result)

ap_dataset_recommended_props_list = [item.asdict() for item in ap_dataset_recommended_props]
create_csv(filepath='csvs/ap-nl-dataset-recommended-props.csv',
           headers=['dataset_prop', 'profile'],
           data_dict=ap_dataset_recommended_props_list
           )   

# Supportive Entities: dcat:Distribution
Dataset recommended prop dcat:distribution 

> A physical embodiment of the Dataset in a particular format. 


Distribution documentation:
* https://semiceu.github.io/DCAT-AP/releases/3.0.1/#Distribution
* https://docs.geostandaarden.nl/dcat/dcat-ap-nl30/#distribution-dcat-distribution

![img/dcatap-NL-DistributionSHACL.svg](img/dcatap-NL-DistributionSHACL.svg) 
image: dcat:Distribution SHACL shapes, based on above query ,rendered by [https://shacl-play.sparna.fr/play/](https://shacl-play.sparna.fr/play/). 




**Mandatory properties:**
* dcat:accessURL (DCAT-AP)
* dct:license (DCAT-AP-NL)

**Interesting (optional) properties for DANS:** 
* dct:issue (data property) The date of formal issuance (e.g., publication)
* http://spdx.org/rdf/terms#checksum (object property) [More on Checksum class](https://semiceu.github.io/DCAT-AP/releases/3.0.1/#Checksum)
    * algorithm = SHA1  (used by Dataverse)
    * checksum value 
* dct:format (object property) - Although [dct:MediaTypeOrExtent](https://www.dublincore.org/specifications/dublin-core/dcmi-terms/#MediaTypeOrExtent) & [dct:MediaType](https://www.dublincore.org/specifications/dublin-core/dcmi-terms/#MediaType) classes to not offer info class properties


In [ ]:

sparql_distrib_shapes = ''' 
DESCRIBE ?shape ?shape_prop
WHERE{
    BIND( dcat:Distribution AS ?targetClass)
    ?shape sh:targetClass ?targetClass ;
            sh:property ?shape_prop .     
}
'''
result_distrib_shapes = sparql_files_query(query=sparql_distrib_shapes,
                                           format='ttl', 
                                           filespath=[
                                               'dcat-ap/releases/3.0.1/shacl/dcat-ap-SHACL.ttl',
                                               'dcat-ap-nl/shapes/dcat-ap-nl-SHACL.ttl'
                                                      ])
print(result_distrib_shapes.serialize(format='ttl').decode('utf-8'))

In [ ]:
# Goal: dcat:Distributions properties ranges



sparql_distrib_props = '''
SELECT DISTINCT  ?prop ?min ?max
WHERE {
        {
            BIND( dcat:Distribution AS ?targetClass)
            ?dist_shape sh:targetClass ?targetClass ; 
                        sh:property ?shape_prop .
            ?shape_prop sh:path ?prop ;
                        sh:name ?prop_name .
            OPTIONAL { ?shape_prop sh:minCount ?min }
            OPTIONAL { ?shape_prop sh:maxCount ?max }

        }

        # UNION
        # {
        #     <https://semiceu.github.io/DCAT-AP/releases/3.0.1/shacl/dcat-ap-SHACL.ttl#dcat:DistributionShape> sh:property ?shape_prop_ .
        #     ?shape_prop_ sh:path ?prop .
        #     FILTER NOT EXISTS { ?shape_prop_ sh:class ?prop_range_ } . 
        # }


            # OPTIONAL {
            #     dcatapnl-sh:DistributionShape sh:property ?nl_shape_prop .
            #     ?nl_shape_prop sh:path ?prop
            #     }
}
ORDER BY ?prop_name
''' 

ap_distrib = sparql_files_query(query=sparql_distrib_props, 
                                                          format='ttl', 
                                                          filespath=['dcat-ap/releases/3.0.1/shacl/dcat-ap-SHACL.ttl',
                                                                     'dcat-ap-nl/shapes/dcat-ap-nl-SHACL.ttl'])
for result in ap_distrib:
    print(result)


# create_csv(filepath='csvs/ap-nl-distribution-overview.csv',
#            headers=['targetClass', 'prop', 'prop_range'],
#            data_dict=[item.asdict() for item in ap_distrib]
#            )   

In [ ]:
# dcat-ap-nl Distribution shapes

describe_distr_sh_NL = '''
DESCRIBE ?shape_prop
WHERE {

           dcatapnl-sh:DistributionShape sh:property ?shape_prop .

}
ORDER BY ?shape_prop
'''


describe_distr_sh_NL_results = sparql_files_query(query=describe_distr_sh_NL, 
                                format='ttl', 
                                filespath=['dcat-ap-nl/shapes/dcat-ap-nl-SHACL.ttl']
                                )
                                           
print(describe_distr_sh_NL_results.serialize(format='ttl').decode('utf-8'))




In [ ]:
# Goal: dcat:Distributions properties ranges
# OUTPUT
# Goal: list of all dcat:Dataset mandatory properties in DCAT-AP & DCAT-APN-NL
# shacl:minCount 1

# https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DistributionShape/0a6f3bb11ed4ea12f852c78996b89c9a54ffc0fb
# https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DistributionShape/73465b7fbd7f991a08ddd1b766c2e46fa9dfc14e 

describe_distr_prop = ''' 
DESCRIBE <https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DistributionShape/0a6f3bb11ed4ea12f852c78996b89c9a54ffc0fb>
<https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DistributionShape/73465b7fbd7f991a08ddd1b766c2e46fa9dfc14e>
'''

describe_distr_prop = '''
DESCRIBE ?shape_prop
WHERE {

            <https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DistributionShape> sh:property ?shape_prop .
            ?shape_prop sh:path ?sh_path ;
                        sh:name ?sh_name .
}
ORDER BY ?sh_name
'''



ap_distrib_ttl = sparql_files_query(query=describe_distr_prop, 
                                format='ttl', 
                                filespath=['dcat-ap/releases/3.0.1/shacl/dcat-ap-SHACL.ttl',]
                                )
                                            #'dcat-ap-nl/shapes/dcat-ap-nl-SHACL.ttl']
                                             


print(ap_distrib_ttl.serialize(format='ttl').decode('utf-8'))






# create_csv(filepath='dcat-ap-nl_mand_props.csv',
#            headers=ap_NL_mandatory_dataset_props_list[0].keys(),
#            data_dict=ap_NL_mandatory_dataset_props_list)

# TODO: dcat-ap/releases/3.0.0/shacl/ranges.ttl


# create_csv(filepath='csvs/ap-nl-distribution-overview.csv',
#            headers=['targetClass', 'prop', 'prop_range'],
#            data_dict=[item.asdict() for item in ap_distrib]
#            )   



<https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DistributionShape/0a6f3bb11ed4ea12f852c78996b89c9a54ffc0fb> rdfs:seeAlso "https://semiceu.github.io/DCAT-AP/releases/3.0.1#Distribution.applicablelegislation";
  shacl:description "The legislation that mandates the creation or management of the Distribution."@en;
  shacl:name "applicable legislation"@en;
  shacl:nodeKind shacl:BlankNodeOrIRI;
  shacl:path <http://data.europa.eu/r5r/applicableLegislation> .



<https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DistributionShape/73465b7fbd7f991a08ddd1b766c2e46fa9dfc14e> rdfs:seeAlso "https://semiceu.github.io/DCAT-AP/releases/3.0.1#Distribution.applicablelegislation";
    shacl:class <http://data.europa.eu/eli/ontology#LegalResource>;
  shacl:description "The legislation that mandates the creation or management of the Distribution."@en;
  shacl:name "applicable legislation"@en;
  shacl:path <http://data.europa.eu/r5r/applicableLegislation> .


In [ ]:
# Output:  for DCAT-AP-NL Distribution mandatory properties and their range (sh:class ?prop_range) 
# Output: ap-nl-dataset-mand-props.csv


sparql_dcat_mandatory_distribution_props_ranges_vars = '''
SELECT  ?prop_path  ?prop_range
WHERE {
    BIND( dcat:Distribution AS ?targetClass )
    VALUES ?prop_path { %s } 
    ?shape a sh:NodeShape ;
           sh:targetClass ?targetClass ;
           sh:property ?prop_shape_uri .
    ?prop_shape_uri sh:path ?prop_path ;
                    sh:class ?prop_range .
    
}''' % prop_path_for_sparql_str4query


ap_NL_mandatory_dataset_props_range = sparql_files_query(query=sparql_dcat_mandatory_distribution_props_ranges_vars, 
                                                          format='ttl', 
                                                          filespath=['dcat-ap/releases/3.0.1/shacl/ranges.ttl'])
ap_NL_mandatory_dataset_props_range_list = [row.asdict() for row in ap_NL_mandatory_dataset_props_range]

# Join  ap_NL_mandatory_dataset_props_range_list & ap_NL_mandatory_dataset_props_list
# Step 1: Build a mapping from prop_path to prop_range
prop_range_map = {d['prop_path']: d['prop_range'] for d in ap_NL_mandatory_dataset_props_range_list}
# Step 2: Merge the lists
ap_NL_mandatory_dataset_props_merged = []
for d in ap_NL_mandatory_dataset_props_list:
    # Copy to avoid mutating the original
    merged_dict = d.copy()
    prop_path = d['prop_path']
    if prop_path in prop_range_map:
        merged_dict['prop_range'] = prop_range_map[prop_path]
    ap_NL_mandatory_dataset_props_merged.append(merged_dict)
print(f"{'*'*3} Output of ap_NL_mandatory_dataset_props_merged in ap-nl-dataset-mand-props.csv {'*'*3}\n")
# print(ap_NL_mandatory_dataset_props_merged)
create_csv(filepath='csvs/ap-nl-dataset-mand-props.csv',
           headers=['prop_path', 'prop_range', 'prop_shape'],
           data_dict=ap_NL_mandatory_dataset_props_merged)


## other focal points:


classes supporting Dataset:
* from prop dct:accessRights  dct:RightsStatement
* from prop dcat:contactPoint vcard:Kind
* from prop dct:creator, dct:publisher foaf:Agent
* from prop dcat:distribution dcat:Distribution (not mandatory, but essential for DANS)

look into foaf:Agent, dcat:Distribution

# Supportive Entities: dct:RightsStatement

The dataset property `dct:accessRights` provide "Information that indicates whether the Dataset is publicly accessible, has access restrictions or is not public. "

```
<https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DatasetShape/ee60fdd4b8581460f5fc4077813b8bb3e3d77441> rdfs:seeAlso "https://semiceu.github.io/DCAT-AP/releases/3.0.1#Dataset.accessrights";
  shacl:class dc:RightsStatement;
  shacl:description "Information that indicates whether the Dataset is publicly accessible, has access restrictions or is not public."@en;
  shacl:name "access rights"@en;
  shacl:path dc:accessRights .
```

**In [DCAT-AP-NL dct:accessRights](https://docs.geostandaarden.nl/dcat/dcat-ap-nl30/#dataset-access-rights) is a mandatory Dataset object property.**
The recommendation from DCAT-AP-NL is to give to provide a value from [Access Rights Named `Authority List](http://publications.europa.eu/resource/authority/access-right). 
Use one of the following values: public (http://publications.europa.eu/resource/authority/access-right/PUBLIC); restricted; non-public.

As all of DANS datasets are public, restriction only happens at the Distribution level and not at the Dataset level. Hence, the simple and correct choice is to make the statement: this dataset has dct:accessRights <http://publications.europa.eu/resource/authority/access-right/PUBLIC>, for all of the datasets.

```ttl
xyz a dcat:Dataset;
    dct:accessRights <http://publications.europa.eu/resource/authority/access-right/PUBLIC>
```


In [ ]:
sparql_access_rights_concepts = '''
SELECT * 
WHERE {?s skos:topConceptOf  <http://publications.europa.eu/resource/authority/access-right> }
'''

access_rights_concepts = sparql_files_query(query=sparql_access_rights_concepts, 
                                                          format='xml', 
                                                          filespath=['https://op.europa.eu/o/opportal-service/euvoc-download-handler?cellarURI=http%3A%2F%2Fpublications.europa.eu%2Fresource%2Fdistribution%2Faccess-right%2F20250924-0%2Frdf%2Fskos_core%2Faccess-right-skos.rdf&fileName=access-right-skos.rdf'])
for concept in access_rights_concepts:
    print(concept.asdict()['s'])

# Supportive Entity: vcard:Kind

Dataset **property dcat:contactPoint target class vcard:Kind**

In [ ]:
# adms publisher type
sparql_access_rights_concepts = '''
SELECT * 
WHERE {<http://purl.org/adms/publishertype/1.0> skos:hasTopConcept  ?publisherType .
        ?publisherType skos:prefLabel ?label . 
}
'''

access_rights_concepts = sparql_files_query(query=sparql_access_rights_concepts, 
                                                          format='xml', 
                                                          filespath=['https://raw.githubusercontent.com/SEMICeu/ADMS-AP/master/purl.org/ADMS_SKOS_v1.00.rdf'])
for concept in access_rights_concepts:
    print(concept.asdict()['label'], concept.asdict()['publisherType'] )
